In [ ]:
import pandas as pd
import numpy as np
import json
import time
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from collections import Counter
import os

In [ ]:
class CFG:
    # данные
    DATA_PATH = "nlp_dataset.csv"
    TEXT_COL  = "clean_text"  
    TARGET    = "Product"   

    # DagsHub / MLflow
    DAGSHUB_OWNER = "kulikanton05"
    DAGSHUB_REPO  = "GP5"

    # train / val / test
    TEST_SIZE    = 0.2  
    VAL_SIZE     = 0.1   
    RANDOM_STATE = 42 

    # обучение DistilBERT
    BERT_MODEL  = "distilbert-base-uncased"
    BERT_MAXLEN = 256
    BERT_EPOCHS = 2
    BERT_BATCH  = 16
    BERT_LR     = 3e-5
    BERT_WEIGHT = 0.01
    
def seed_everything(seed=CFG.RANDOM_STATE):
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything()

In [5]:
df = pd.read_csv(CFG.DATA_PATH)
df

,Consumer complaint narrative,Product,Issue,word_count,clean_text
0,I mobile deposited a XXXX check to Ameris Bank...,Checking or savings account,Managing an account,188,i mobile deposited a check to ameris bank on y...
1,"Credence Resource Management , LLC ( Use the a...",Debt collection,Attempts to collect debt not owed,332,credence resource management llc use the addre...
2,"Help my rights are being violated, 15 U.S.C. 1...",Debt collection,Took or threatened to take negative or legal a...,85,help my rights are being violated u s c g b of...
3,This company is violating my rights they have ...,Debt collection,Took or threatened to take negative or legal a...,74,this company is violating my rights they have ...
4,My information was heisted and this accounts a...,Credit card,Other,31,my information was heisted and this accounts a...
...,...,...,...,...,...
53893,My credit reports are inaccurate. These inaccu...,Credit reporting or other personal consumer re...,Incorrect information on your report,39,my credit reports are inaccurate these inaccur...
53894,I received a very shady voicemail from structu...,Debt collection,Other,39,i received a very shady voicemail from structu...
53895,In XXXX of XXXX I was charged for a Best Buy r...,Credit card,Problem with a purchase shown on your statement,434,in of i was charged for a best buy renewal of ...
53896,"My name is XXXX XXXX, residing at XXXX XXXX XX...",Debt collection,Attempts to collect debt not owed,277,my name is residing at ga my date of birth is ...


In [6]:
df = df.reset_index(drop=True)
df["row_id"] = df.index
df

,Consumer complaint narrative,Product,Issue,word_count,clean_text,row_id
0,I mobile deposited a XXXX check to Ameris Bank...,Checking or savings account,Managing an account,188,i mobile deposited a check to ameris bank on y...,0
1,"Credence Resource Management , LLC ( Use the a...",Debt collection,Attempts to collect debt not owed,332,credence resource management llc use the addre...,1
2,"Help my rights are being violated, 15 U.S.C. 1...",Debt collection,Took or threatened to take negative or legal a...,85,help my rights are being violated u s c g b of...,2
3,This company is violating my rights they have ...,Debt collection,Took or threatened to take negative or legal a...,74,this company is violating my rights they have ...,3
4,My information was heisted and this accounts a...,Credit card,Other,31,my information was heisted and this accounts a...,4
...,...,...,...,...,...,...
53893,My credit reports are inaccurate. These inaccu...,Credit reporting or other personal consumer re...,Incorrect information on your report,39,my credit reports are inaccurate these inaccur...,53893
53894,I received a very shady voicemail from structu...,Debt collection,Other,39,i received a very shady voicemail from structu...,53894
53895,In XXXX of XXXX I was charged for a Best Buy r...,Credit card,Problem with a purchase shown on your statement,434,in of i was charged for a best buy renewal of ...,53895
53896,"My name is XXXX XXXX, residing at XXXX XXXX XX...",Debt collection,Attempts to collect debt not owed,277,my name is residing at ga my date of birth is ...,53896


In [7]:
le = LabelEncoder()
y = le.fit_transform(df[CFG.TARGET])
X = df[CFG.TEXT_COL]

In [8]:
dict(zip(le.classes_, le.transform(le.classes_)))

{'Checking or savings account': np.int64(0),
 'Credit card': np.int64(1),
 'Credit reporting or other personal consumer reports': np.int64(2),
 'Debt collection': np.int64(3),
 'Money transfer, virtual currency, or money service': np.int64(4)}

In [9]:
class_names = list(le.classes_)
num_classes = len(class_names)
class_names, num_classes

(['Checking or savings account',
  'Credit card',
  'Credit reporting or other personal consumer reports',
  'Debt collection',
  'Money transfer, virtual currency, or money service'],
 5)

In [10]:
X_trv, X_test, y_trv, y_test = train_test_split(X, y, test_size=CFG.TEST_SIZE, random_state=CFG.RANDOM_STATE, stratify=y)
val_rel = CFG.VAL_SIZE/ (1.0 - CFG.TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(X_trv, y_trv, test_size=val_rel, random_state=CFG.RANDOM_STATE, stratify=y_trv)

In [11]:
len(X_train), len(X_val), len(X_test)

(37728, 5390, 10780)

In [12]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

cuda


In [ ]:
def save_split_csv(X, y, part_name):
    out = pd.DataFrame({
        CFG.TEXT_COL: X,
        "label": y,
        CFG.TARGET: le.inverse_transform(y)
    })
    path = rf'D:\Alexander\Desktop\ГП5 доп\artifacts\{part_name}.csv'
    out.to_csv(path, index=False)
    return path

SPLIT_PATHS = {
    "train": save_split_csv(X_train, y_train, "train"),
    "val":   save_split_csv(X_val,   y_val,   "val"),
    "test":  save_split_csv(X_test,  y_test,  "test"),
}
print("Сохранены сплиты:", SPLIT_PATHS)

Сохранены сплиты: {'train': 'D:\\Alexander\\Desktop\\ГП5 доп\\artifacts\\train.csv', 'val': 'D:\\Alexander\\Desktop\\ГП5 доп\\artifacts\\val.csv', 'test': 'D:\\Alexander\\Desktop\\ГП5 доп\\artifacts\\test.csv'}


In [20]:
import dagshub
import mlflow

dagshub.init(repo_owner=CFG.DAGSHUB_OWNER, repo_name=CFG.DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment("Product_Classification")
print("Tracking URI:", mlflow.get_tracking_uri())

d:\Alexander\Desktop\ГП5 доп\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as NazarovAlexander06

Initialized MLflow to track repo "kulikanton05/GP5"

Repository kulikanton05/GP5 initialized!

Tracking URI: https://dagshub.com/kulikanton05/GP5.mlflow


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

bert_tokenizer = AutoTokenizer.from_pretrained(CFG.BERT_MODEL)
class BertTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=CFG.BERT_MAXLEN):
        self.enc = tokenizer([str(t) for t in texts], truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(np.asarray(labels, dtype=np.int64))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

bert_train_loader = DataLoader(BertTextDataset(X_train, y_train, bert_tokenizer), batch_size=CFG.BERT_BATCH, shuffle=True)
bert_val_loader   = DataLoader(BertTextDataset(X_val,   y_val,   bert_tokenizer), batch_size=CFG.BERT_BATCH, shuffle=False)
bert_test_loader  = DataLoader(BertTextDataset(X_test,  y_test,  bert_tokenizer), batch_size=CFG.BERT_BATCH, shuffle=False)


In [ ]:
def bert_train_epoch(model, loader, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    preds, trues = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
        out.loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += out.loss.item() * labels.size(0)
        n += labels.size(0)
        preds.extend(out.logits.argmax(1).detach().cpu().tolist())
        trues.extend(labels.detach().cpu().tolist())

    m = compute_metrics(trues, preds); m["loss"] = total_loss / max(n, 1)
    return m

@torch.no_grad()
def bert_validate_epoch(model, loader):
    model.eval()
    total_loss, n = 0.0, 0
    preds, trues = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
        total_loss += out.loss.item() * labels.size(0)
        n += labels.size(0)
        preds.extend(out.logits.argmax(1).detach().cpu().tolist())
        trues.extend(labels.detach().cpu().tolist())
    m = compute_metrics(trues, preds); m["loss"] = total_loss / max(n, 1)
    return m

@torch.no_grad()
def bert_evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        out = model(input_ids=input_ids, attention_mask=attn)
        preds.extend(out.logits.argmax(1).detach().cpu().tolist())
        trues.extend(batch["labels"].tolist())
    return compute_metrics(trues, preds), trues, preds


In [24]:
seed_everything()
bert_model = AutoModelForSequenceClassification.from_pretrained(CFG.BERT_MODEL, num_labels=num_classes).to(device)
bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=CFG.BERT_LR)

bert_history = []
for epoch in range(1, CFG.BERT_EPOCHS + 1):
    t0 = time.time()
    tr = bert_train_epoch(bert_model, bert_train_loader, bert_optimizer)
    va = bert_validate_epoch(bert_model, bert_val_loader)
    bert_history.append({
        "train_loss": tr["loss"], "val_loss": va["loss"],
        "val_accuracy": va["accuracy"], "val_f1_macro": va["f1_macro"],
    })
    print(f"[DistilBERT] epoch {epoch}/{CFG.BERT_EPOCHS} | {time.time()-t0:4.1f}s "
          f"| train_loss {tr['loss']:.4f} | val_loss {va['loss']:.4f} "
          f"| val_acc {va['accuracy']:.3f} | val_f1 {va['f1_macro']:.3f}")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6675.21it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[DistilBERT] epoch 1/2 | 892.1s | train_loss 0.4731 | val_loss 0.3702 | val_acc 0.870 | val_f1 0.831
[DistilBERT] epoch 2/2 | 896.9s | train_loss 0.3263 | val_loss 0.3791 | val_acc 0.868 | val_f1 0.831


In [25]:
bert_test, bert_true, bert_pred = bert_evaluate(bert_model, bert_test_loader)
print("DistilBERT test metrics:", {k: round(v, 4) for k, v in bert_test.items()})
print(classification_report(bert_true, bert_pred, target_names=class_names, zero_division=0))

DistilBERT test metrics: {'accuracy': 0.8704, 'precision_macro': 0.836, 'recall_macro': 0.832, 'f1_macro': 0.8322}
                                                     precision    recall  f1-score   support

                        Checking or savings account       0.77      0.89      0.82      2334
                                        Credit card       0.86      0.84      0.85      2275
Credit reporting or other personal consumer reports       0.83      0.86      0.84       542
                                    Debt collection       0.96      0.93      0.95      4629
 Money transfer, virtual currency, or money service       0.76      0.65      0.70      1000

                                           accuracy                           0.87     10780
                                          macro avg       0.84      0.83      0.83     10780
                                       weighted avg       0.87      0.87      0.87     10780



In [27]:
with mlflow.start_run(run_name="DistilBERT"):
    mlflow.log_params({
        "model_name":   "DistilBERT",
        "target":       CFG.TARGET,
        "batch_size":   CFG.BERT_BATCH,
        "epochs":       CFG.BERT_EPOCHS,
        "learning_rate": CFG.BERT_LR,
        "base_model":   CFG.BERT_MODEL,
        "max_len":      CFG.BERT_MAXLEN,
    })

    for ep, h in enumerate(bert_history, start=1):
        mlflow.log_metrics({
            "train_loss":   h["train_loss"],
            "val_loss":     h["val_loss"],
            "val_accuracy": h["val_accuracy"],
            "val_f1_macro": h["val_f1_macro"],
        }, step=ep)

    mlflow.log_metrics({
        "accuracy":        bert_test["accuracy"],
        "precision_macro": bert_test["precision_macro"],
        "recall_macro":    bert_test["recall_macro"],
        "f1_macro":        bert_test["f1_macro"],
    })

    report = classification_report(bert_true, bert_pred, target_names=class_names, zero_division=0)
    rep_path = rf"D:\Alexander\Desktop\ГП5 доп\artifacts\DistilBERT_report.txt"
    with open(rep_path, "w") as f:
        f.write(report)
    mlflow.log_artifact(rep_path, artifact_path="reports")

    save_dir = rf"D:\Alexander\Desktop\ГП5 доп\artifacts\distilbert_hf"
    os.makedirs(save_dir, exist_ok=True)
    bert_model.save_pretrained(save_dir)
    bert_tokenizer.save_pretrained(save_dir)
    with open(os.path.join(save_dir, "label_classes.json"), "w") as f:
        json.dump(class_names, f, ensure_ascii=False)
    mlflow.log_artifacts(save_dir, artifact_path="model")

    for p in SPLIT_PATHS.values():
        mlflow.log_artifact(p, artifact_path="data")

print("[DistilBERT] залогирован в MLflow.")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.99it/s]


🏃 View run DistilBERT at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/0/runs/8fee02b2893b4192a575fdd04559c429
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/0
[DistilBERT] залогирован в MLflow.


In [ ]:
@torch.no_grad()
def predict_bert_texts(model, tokenizer, texts, max_len, batch_size):
    model.eval()
    all_preds = []
    all_probs = []
    texts = [str(t) for t in texts]
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(batch_texts, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        conf, preds = torch.max(probs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(conf.cpu().numpy())
    return np.array(all_preds), np.array(all_probs)

In [29]:
product_pred_ids, product_conf = predict_bert_texts(
    model=bert_model,
    tokenizer=bert_tokenizer,
    texts=df[CFG.TEXT_COL],
    max_len=CFG.BERT_MAXLEN,
    batch_size=CFG.BERT_BATCH
)

df["predicted_product"] = le.inverse_transform(product_pred_ids)
df["product_confidence"] = product_conf

In [ ]:
product_predictions = df[[
    "row_id",
    "clean_text",
    "Product",
    "predicted_product",
    "product_confidence"
]].copy()

product_predictions = product_predictions.rename(columns={"Product": "true_product"})
product_predictions.to_csv("product_predictions.csv", index=False)
product_predictions.head()

,row_id,clean_text,true_product,predicted_product,product_confidence
0,0,i mobile deposited a check to ameris bank on y...,Checking or savings account,Checking or savings account,0.977000
1,1,credence resource management llc use the addre...,Debt collection,Debt collection,0.994181
2,2,help my rights are being violated u s c g b of...,Debt collection,Debt collection,0.996244
3,3,this company is violating my rights they have ...,Debt collection,Debt collection,0.997443
4,4,my information was heisted and this accounts a...,Credit card,Credit card,0.989423
